# Дообучение модели на вашей переписке

Бесплатно в Google Colab: **Среда выполнения → Сменить среду выполнения → GPU T4**.

Базовая модель — открытая Qwen2.5-7B-Instruct (хорошо понимает русский). Обучаем LoRA-адаптер
в 4 бита (QLoRA), затем сохраняем модель в GGUF, чтобы запускать её у себя через Ollama.

Запускайте ячейки по порядку.

In [ ]:
!pip install -q unsloth

In [ ]:
# Загрузите data/train.jsonl, созданный prepare_dataset.py
from google.colab import files
files.upload()

In [ ]:
# Если на вашем компьютере меньше 8 ГБ ОЗУ — возьмите unsloth/Qwen2.5-3B-Instruct-bnb-4bit
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
# Ограничение, чтобы обучение уложилось в лимит бесплатного Colab (~1-2 часа)
MAX_SAMPLES = 5000

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
dataset = dataset.map(
    lambda batch: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]},
    batched=True,
    remove_columns=dataset.column_names,
)
print(len(dataset), "примеров")
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        output_dir="outputs",
        report_to="none",
        seed=3407,
    ),
)
# Учим модель только на ваших ответах, а не на сообщениях собеседников
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

In [ ]:
# Быстрая проверка: как модель ответит (замените имена на свои)
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "Ты — Андрей. Ты переписываешься в Telegram с собеседником по имени Дима. "
     "Отвечай в стиле Андрей: тем же тоном, длиной сообщений, словами и пунктуацией. "
     "Если отвечаешь несколькими сообщениями подряд — пиши каждое с новой строки."},
    {"role": "user", "content": "привет, че делаешь вечером?"},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
output = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Сохраняем в GGUF (≈4.7 ГБ для 7B) и копируем на Google Диск — так надёжнее, чем скачивать из браузера
model.save_pretrained_gguf("twin", tokenizer, quantization_method="q4_k_m")

import glob, shutil
from google.colab import drive
drive.mount("/content/drive")
for path in glob.glob("**/*.gguf", recursive=True):
    print("Копирую", path)
    shutil.copy(path, "/content/drive/MyDrive/")
print("Готово: скачайте .gguf с Google Диска и переименуйте в twin.gguf")